In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

In [14]:
X, y = make_classification(
    n_samples=2500,       
    n_features=20,        
    n_informative=12,      
    n_classes=2,          
    random_state=8       
)

In [15]:
X[:2]

array([[ 0.66393907, -0.08151312, -0.35574504, -2.25523723, -0.82459016,
         0.45469416, -0.24020224, -0.01301753, -2.15150055, -0.20909669,
         1.43943625,  2.27445866,  1.98966981, -0.06946088,  0.07519195,
        -1.39202067,  0.40889453,  1.00180758, -2.70553892,  1.53922232],
       [ 1.93832827,  0.32664886,  1.48643337, -2.02639728, -1.95836646,
         4.80250341,  3.01283631,  0.0772098 ,  1.17200786, -3.12323417,
         0.34412466, -2.89454636,  0.18859416,  6.6140513 , -0.51655982,
         4.92702996, -9.3346991 ,  2.85158828,  0.81600583, -0.58270121]])

In [16]:
y[:2]

array([1, 1])

In [17]:
X.shape

(2500, 20)

In [18]:
y.shape

(2500,)

In [19]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=8)


In [111]:
X_train.shape

(2000, 20)

# Partie 1 : Bagging & Random Forest

## 1.1. Modèle de base

In [20]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

In [29]:
model = DecisionTreeClassifier(random_state=8)

# 5. Train 
model.fit(X_train, y_train)

# 6. predictions
predictions = model.predict(X_test)

# 7. Evaluation
accuracy = accuracy_score(y_test, predictions)
print(f"Model Accuracy: {accuracy * 100:.2f}%")

Model Accuracy: 83.20%


In [30]:


# 7. Evaluation sur l'ensemble d'entrainement

print(f"Model Accuracy: { accuracy_score(y_train, model.predict(X_train))* 100:.2f}%")

Model Accuracy: 100.00%


## 1.2. Implémentation manuelle du Bagging

In [31]:
2500/50

50.0

In [92]:
pd.DataFrame(X).sample(50).index

Index([1315, 1626,  614, 1687, 1554,   69,  687,  234,  360, 1701,   92, 1032,
       1463,  808, 1744, 2096, 1815, 2046, 1241,  996, 1683, 1923,  861,  558,
        877,  916,  165,  535,  838, 2437, 2249, 2070,  751, 1624,  790, 2176,
        835, 1722,  763, 2144, 2440, 1142, 1574, 1761,  133, 2019, 1565, 1368,
       1404,  982],
      dtype='int64')

In [90]:
pd.DataFrame(X).sample(50).index.shape

(50,)

In [106]:
bootstraps_index = [pd.DataFrame(X_train).sample(50).index for i in range(50)]    

In [98]:
np.shape(bootstraps_index)

(50, 50)

In [119]:
ens_preds = []
for i, index in enumerate(bootstraps_index):
    
    model = DecisionTreeClassifier(random_state=68 + i) 
    
    
    X_boot = pd.DataFrame(X_train).iloc[index]
    y_boot = pd.Series(y_train).iloc[index]
    
    
    model.fit(X_boot, y_boot)
    y_pred = model.predict(X_test)
    ens_preds.append(y_pred)

In [121]:
pd.DataFrame(ens_preds).value_counts()[:2]

0  1  2  3  4  5  6  7  8  9  10  11  12  13  14  15  16  17  18  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35  36  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53  54  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71  72  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88  89  90  91  92  93  94  95  96  97  98  99  100  101  102  103  104  105  106  107  108  109  110  111  112  113  114  115  116  117  118  119  120  121  122  123  124  125  126  127  128  129  130  131  132  133  134  135  136  137  138  139  140  141  142  143  144  145  146  147  148  149  150  151  152  153  154  155  156  157  158  159  160  161  162  163  164  165  166  167  168  169  170  171  172  173  174  175  176  177  178  179  180  181  182  183  184  185  186  187  188  189  190  191  192  193  194  195  196  197  198  199  200  201  202  203  204  205  206  207  208  209  210  211  212  213  214  215  216  217  218  219  220  221  

In [125]:
len(pd.DataFrame(ens_preds).value_counts().idxmax())

500

In [126]:
pd.DataFrame(ens_preds).value_counts().idxmax()[:3]

(np.int64(1), np.int64(0), np.int64(1))

In [ ]:
df_preds = pd.DataFrame(ens_preds).T


predictions_finales = df_preds.mode(axis=1)[0]

print(len(predictions_finales)) 

500


## 1.3. Random Fores